In [1]:
import numpy as np
import pandas as pd
import nltk

In [2]:
dataset = pd.read_csv("spam.csv", encoding="latin-1")

In [3]:
dataset.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
dataset = dataset.drop(columns=["Unnamed: 2","Unnamed: 3","Unnamed: 4"])

In [5]:
dataset = dataset.rename(columns={"v1":"target", "v2":"text"})

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [7]:
dataset["target"] = le.fit_transform(dataset["target"]) 

In [8]:
dataset.isnull().sum()

target    0
text      0
dtype: int64

In [9]:
dataset.duplicated().sum()

np.int64(403)

In [10]:
dataset = dataset.drop_duplicates(keep="first")

## data per

In [11]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.corpus import stopwords
stopwords = stopwords.words("english")

import string
punctuation = string.punctuation

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [13]:
def trans_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    
    y=[]
    """for i in text:
        if i.isalnum():
            y.append(i)"""

    #text = y[:]   #copy
    #y.clear()     #clear
    for i in text:
        if i not in stopwords and i not in punctuation and i.isalnum() :
            y.append(i)
    text=y[:]
    y.clear()
    for i in text :
        y.append(ps.stem(i))
    return " ".join(y)

In [14]:
dataset["transformed_text"] = dataset["text"].apply(trans_text)

In [15]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
cv  = CountVectorizer()
tfidf = TfidfVectorizer()

In [16]:
X = tfidf.fit_transform(dataset["transformed_text"]).toarray()

In [17]:
Y = dataset["target"]

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [19]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB

In [20]:
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [21]:
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()

In [22]:
gnb.fit(x_train,y_train)
y_pred1 = gnb.predict(x_test)
print(accuracy_score(y_test,y_pred1))
print(confusion_matrix(y_test,y_pred1))
print(precision_score(y_test,y_pred1))

0.8636363636363636
[[772 117]
 [ 24 121]]
0.5084033613445378


In [23]:
mnb.fit(x_train,y_train)
y_pred2 = mnb.predict(x_test)
print(accuracy_score(y_test,y_pred2))
print(confusion_matrix(y_test,y_pred2))
print(precision_score(y_test,y_pred2))

0.9613152804642167
[[888   1]
 [ 39 106]]
0.9906542056074766


In [24]:
bnb.fit(x_train,y_train)
y_pred3 = bnb.predict(x_test)
print(accuracy_score(y_test,y_pred3))
print(confusion_matrix(y_test,y_pred3))
print(precision_score(y_test,y_pred3))

0.9661508704061895
[[885   4]
 [ 31 114]]
0.9661016949152542


In [28]:
#import pickle
#pickle.dump(tfidf,open("vector.pkl","wb"))

In [29]:
#pickle.dump(mnb,open("spam_model.pkl","wb"))